In [1]:
import re
from collections import Counter
from pathlib import Path

import yaml
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

KB_DIR = Path("../data/knowledge_base/generated")   # notebook runs from RAG_pipeline/
CITIES = ["Delhi", "Pune", "Kolkata"]               # first iteration: 3 cities only
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
CHUNK_TOKENS = 450                                  # bge-small reads max 512 tokens
OVERLAP_TOKENS = 50


In [ ]:
def load_md(path):
    raw = path.read_text(encoding="utf-8")
    _, front, body = raw.split("---", 2)            # file starts with ---\n<yaml>\n---
    meta = yaml.safe_load(front)

    # Chroma accepts only str / int / float / bool (YAML turns 2024-05-01 into a date)
    meta = {k: v if isinstance(v, (str, int, float, bool)) else str(v) for k, v in meta.items()}
    meta.pop("source", None)                        # same long string in every file

    body = body.strip()
    meta["path"] = str(path.relative_to(KB_D  IR))
    meta["title"] = re.search(r"^# (.+)$", body, re.M).group(1)
    return {"text": body, "metadata": meta}


In [3]:
docs = []
for path in sorted(KB_DIR.rglob("*.md")):
    doc = load_md(path)
    city = doc["metadata"].get("city")
    if city in CITIES or doc["metadata"]["doc_type"] in ("dataset_card", "reference"):
        docs.append(doc)

print("Documents loaded:", len(docs))
print(Counter(d["metadata"].get("city", "(no city)") for d in docs))
print(Counter(d["metadata"]["doc_type"] for d in docs))


Documents loaded: 528
Counter({'Delhi': 175, 'Kolkata': 175, 'Pune': 175, '(no city)': 3})
Counter({'monthly_summary': 360, 'seasonal_summary': 117, 'event_log': 33, 'city_profile': 3, 'event_summary': 3, 'rainy_season': 3, 'records': 3, 'trends': 3, 'reference': 2, 'dataset_card': 1})


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=CHUNK_TOKENS,
    chunk_overlap=OVERLAP_TOKENS,
    separators=["\n## ", "\n### ", "\n\n", "\n"],
)

chunks = []
for doc in docs:
    title = doc["metadata"]["title"]
    for i, text in enumerate(splitter.split_text(doc["text"])):
        if not text.startswith("# "):               # later pieces: say which city/year they belong to
            text = f"# {title}\n\n{text}"
        chunks.append({
            "id": f"{doc['metadata']['path']}#{i}",
            "text": text,
            "metadata": {**doc["metadata"], "chunk": i},
        })

print("Chunks:", len(chunks))


Chunks: 546


In [6]:
from langchain_core.documents import Document

documents = [Document(page_content=c["text"], metadata=c["metadata"], id=c["id"]) for c in chunks]
print("LangChain documents:", len(documents))


LangChain documents: 546


In [7]:
from langchain_core.documents import Document

documents = [Document(page_content=c["text"], metadata=c["metadata"], id=c["id"]) for c in chunks]
print("LangChain documents:", len(documents))


LangChain documents: 546


In [8]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

embedding_model = HuggingFaceBgeEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
    query_instruction="Represent this sentence for searching relevant passages: ",
)


/tmp/ipykernel_216523/2313592300.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceBgeEmbeddings
/tmp/ipykernel_216523/2313592300.py:3: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceBgeEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
from langchain_chroma import Chroma
import shutil, os, time

VECTOR_DIR = "../vectorstore/chroma"
COLLECTION = "weather_kb"

if os.path.isdir(VECTOR_DIR):
    shutil.rmtree(VECTOR_DIR)     # rebuild fresh every run

t = time.time()
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name=COLLECTION,
    persist_directory=VECTOR_DIR,
    collection_metadata={"hnsw:space": "cosine"},
)
print("Stored:", vectorstore._collection.count(), f"{time.time()-t:.0f}s")


Stored: 546 69s


In [ ]:
import re

MONTHS = {"january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6, "july": 7,
          "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
          "jan": 1, "feb": 2, "mar": 3, "apr": 4, "jun": 6, "jul": 7, "aug": 8, "sep": 9, "sept": 9,
          "oct": 10, "nov": 11, "dec": 12}

def extract_filters(question):
    q = question.lower()
    conditions = []

    for city in CITIES:
        if city.lower() in q:
            conditions.append({"city": city})
            break  # assumtion one city per question

    year_match = re.search(r"\b(20(?:1[5-9]|2[0-5]))\b", q)
    if year_match:
        conditions.append({"year": int(year_match.group(1))})

    for name, num in MONTHS.items():
        if re.search(rf"\b{name}\b", q):
            conditions.append({"month": num})
            break

    if len(conditions) == 0:
        return None
    if len(conditions) == 1:
        return conditions[0]       # Chroma wants $and only for 2+ conditions
    return {"$and": conditions}


In [ ]:
import os
from google import genai

GEMINI_MODEL = "gemini-3.5-flash-lite"   # free-tier Flash model


SYSTEM_PROMPT = """You are a weather assistant for Indian cities, answering from the context below only.

Rules:
- Use ONLY the facts in the context. Do not use outside knowledge.
- If the context does not answer the question, say "I don't have that information in my data."
- Always cite the city and the exact date(s) or month/year your answer is based on.
- Be concise.
"""

def build_prompt(question, retrieved):
    context = "\n\n---\n\n".join(doc.page_content for doc, _ in retrieved)
    return f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"


In [ ]:
def answer(question, k=3):
    where = extract_filters(question)
    retrieved = vectorstore.similarity_search_with_score(question, k=k, filter=where)

    prompt = build_prompt(question, retrieved)
    response = gemini_client.models.generate_content(model=GEMINI_MODEL, contents=prompt)

    print(f"Q: {question}")
    print(f"Filter used: {where}")
    print(f"Sources: {[doc.metadata['path'] for doc, _ in retrieved]}")
    print(f"\nA: {response.text}")
    print("=" * 100)
    return response.text


In [ ]:
answer("How hot was Delhi in May 2024?")
answer("When did Kolkata have a heatwave in 2024?")
answer("Is Pune getting hotter over the years?")
answer("What is the capital of France?")   # should refuse — out of scope


Q: How hot was Delhi in May 2024?
Filter used: {'$and': [{'city': 'Delhi'}, {'year': 2024}, {'month': 5}]}
Sources: ['monthly/Delhi/Delhi_2024_05.md']

A: Based on Delhi in May 2024, the average maximum temperature was 41.7°C (which was 2.8°C warmer than the May normal of 38.9°C), and the highest max temperature reached 46.0°C on 2024-05-26.
Q: When did Kolkata have a heatwave in 2024?
Filter used: {'$and': [{'city': 'Kolkata'}, {'year': 2024}]}
Sources: ['events/Kolkata/Kolkata_2024_events.md', 'monthly/Kolkata/Kolkata_2024_12.md', 'monthly/Kolkata/Kolkata_2024_07.md']

A: In Kolkata, heatwaves occurred during the following periods in 2024:
- 2024-04-19 to 2024-04-21
- 2024-04-25 to 2024-05-03
Q: Is Pune getting hotter over the years?
Filter used: {'city': 'Pune'}
Sources: ['trends/Pune_trends.md', 'city_profiles/Pune.md', 'monthly/Pune/Pune_2021_09.md']

A: Based on the 2015–2024 data for Pune, a straight-line fit shows no clear change in the average daily maximum temperature.
Q: Wha

"I don't have that information in my data."